<a href="https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane: **classification**. Same mid-panel month as ML-04/05/06 (`month=2026-03`). This is the transparent, hand-readable rule that any Week-5 model must beat — a score with **one** reason code and an action label, checked against two real signals first.

> The `work/outputs/baseline_action_score.csv` this notebook writes is intentionally **not committed** — the CI leak-guard blocks data files, and the notebook regenerates it on every run. Only the notebook itself gets committed.

## Setup — connect to the warehouse (Hugging Face via DuckDB)

In [12]:
%pip install -q duckdb huggingface_hub pandas numpy

import duckdb, os
import pandas as pd
import numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")

MONTH = "2026-03"
TABLE_URI = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"

# Content-month grain, same filter established in ML-04/05/06 (only ~36.7% of rows have real GSC data)
df = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks)       AS total_clicks,
        SUM(gsc_impressions)  AS total_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr
    FROM read_parquet('{TABLE_URI}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

df["avg_ctr"] = df["avg_ctr"].fillna(0)

position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21-50", "51+"]
df["position_tier"] = pd.cut(df["avg_position"], bins=position_bins, labels=position_labels)

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738


,content_hash_id,client_hash_id,avg_position,total_clicks,total_impressions,avg_ctr,position_tier
0,content_05597932fe4da067,client_73cda7b4e4f265ea,2.714744,0.0,57.0,0.000000,1-3
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,7.209549,7.0,6523.0,0.001073,4-10
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,6.481453,0.0,149.0,0.000000,4-10
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2.987198,0.0,453.0,0.000000,1-3
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,6.724039,6.0,5630.0,0.001066,4-10


## 1. My rule and its reason codes

**The rule, in plain words:** A page is worth reviewing if it already gets meaningful search visibility (impressions), but its click-through rate is below what pages at its own position tier typically achieve. That gap — visible, but underperforming for its rank — is a concrete, fixable signal (title/snippet/meta issue), not a ranking problem. The score is the size of that gap weighted by how much impression volume is being wasted.

Before encoding it, two signals it leans on are checked below — one bucket table each, with **n** printed, and a one-word verdict.

In [13]:
# --- Signal check 1 (flag-linked: behind the CTR-fix logic) ---
signal1 = df.groupby("position_tier", observed=True).agg(
    weighted_ctr=("total_clicks", lambda s: s.sum()),
    total_impr=("total_impressions", "sum"),
    n=("total_impressions", "count")
).reset_index()
signal1["weighted_ctr"] = signal1["weighted_ctr"] / signal1["total_impr"]
signal1["insufficient_data"] = signal1["n"] < 50
print("Signal 1 — position tier vs weighted CTR (sum clicks / sum impressions):")
print(signal1[["position_tier", "weighted_ctr", "n", "insufficient_data"]])

Signal 1 — position tier vs weighted CTR (sum clicks / sum impressions):
  position_tier  weighted_ctr      n  insufficient_data
0           1-3      0.004063  16144              False
1          4-10      0.003239  81987              False
2         11-20      0.003052  32204              False
3         21-50      0.001366  33288              False
4           51+      0.000392  11681              False


Weighted CTR by position tier: 1-3: 0.4063%, 4-10: 0.3239%, 11-20: 0.3052%, 21-50: 0.1366%, 51+: 0.0392%. n per tier: 16144 / 81987 / 32204 / 33288 / 11681 (all well above the 50-row floor). Verdict: CONFIRMED — the CTR-fix flag's assumption is supported, because CTR declines monotonically and substantially as position gets worse (over 10x lower at tier 51+ than at tier 1-3).

In [14]:
# --- Signal check 2 (flag-linked: behind the quick-win logic) ---
# Claim: "High-impression pages that rank outside the top 10 have below-average CTR" —
# the assumption the quick-win flag leans on (volume is there, position is fixable, CTR is being left on the table).
impression_floor = df["total_impressions"].median()
df["quick_win_candidate"] = np.where(
    (df["total_impressions"] >= impression_floor) & (~df["position_tier"].isin(["1-3", "4-10"])),
    "high_volume_outside_top10", "other"
)

signal2 = df.groupby("quick_win_candidate", observed=True)["avg_ctr"].agg(median="median", n="count").reset_index()
signal2["insufficient_data"] = signal2["n"] < 50
print("Signal 2 — quick-win candidates (high impressions, outside top 10) vs median CTR:")
print(signal2)

Signal 2 — quick-win candidates (high impressions, outside top 10) vs median CTR:
         quick_win_candidate    median       n  insufficient_data
0  high_volume_outside_top10  0.000584   37670              False
1                      other  0.000000  139068              False


Median CTR, high-volume-outside-top10 vs other: 0.000584 vs 0.000000. n: 37,670 vs 139,068 (both well above floor). Verdict: MIXED — the quick-win flag's assumption is only partially supported: the "high volume outside top 10" group does show a nonzero median CTR while the rest of the population is swamped by zero-click pages, so directionally the story holds. But the effect size is small in absolute terms (0.06%), and the comparison group's median being exactly 0 (from zero-inflation, confirmed in ML-06) makes this specific median-based test weaker evidence than Signal 1's weighted-CTR approach. A clearly-explained negative here is still a win — it means the rule below should lean more heavily on Signal 1's position-tier pattern than on this specific quick-win split.

## 2. Build the ranked queue (writes the CSV)

One score, **one** reason code, one action label — readable on purpose, no fitted weights. Only uses fields already knowable within the same month (no future window, no label-derived input — there's no label in this baseline at all, it's a rule, not a model).

In [15]:
# Expected CTR per position tier, from Signal 1's own bucket weighted CTR — this IS the transparent
# "what should this page be getting" baseline the rule compares each page against.
tier_agg = df.groupby("position_tier", observed=True).agg(
    tier_clicks=("total_clicks", "sum"),
    tier_impressions=("total_impressions", "sum")
)
expected_ctr_by_tier = (tier_agg["tier_clicks"] / tier_agg["tier_impressions"]).astype(float)
expected_ctr_by_tier.index = expected_ctr_by_tier.index.astype(str)

df["expected_ctr"] = df["position_tier"].astype(str).map(expected_ctr_by_tier).astype(float)

df["ctr_gap"] = (df["expected_ctr"] - df["avg_ctr"]).clip(lower=0)
visible = (df["total_impressions"] >= impression_floor).astype(int)

# Readable on purpose: visibility gate x underperformance gap x impression volume wasted
df["score"] = visible * df["ctr_gap"] * df["total_impressions"]

REASON_CODE = "CTR_GAP_VS_POSITION_TIER"
df["reason_code"] = np.where(df["score"] > 0, REASON_CODE, "no_gap")
df["action_label"] = np.where(df["score"] > 0, "review_ctr", "no_action")

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote", len(ranked), "rows to work/outputs/baseline_action_score.csv")
print("Rows with an action (score > 0):", (ranked["score"] > 0).sum())
ranked[["content_hash_id", "client_hash_id", "position_tier", "total_impressions",
         "avg_ctr", "expected_ctr", "ctr_gap", "score", "reason_code", "action_label"]].head(10)

Wrote 176738 rows to work/outputs/baseline_action_score.csv
Rows with an action (score > 0): 60030


,content_hash_id,client_hash_id,position_tier,total_impressions,avg_ctr,expected_ctr,ctr_gap,score,reason_code,action_label
0,content_44f34c0a90047651,client_23a62021009f63c4,4-10,212404.0,0.000113,0.003239,0.003126,663.966533,CTR_GAP_VS_POSITION_TIER,review_ctr
1,content_8d7d99f109e19aa2,client_e547b89c05043229,1-3,203497.0,0.001420,0.004063,0.002642,537.712743,CTR_GAP_VS_POSITION_TIER,review_ctr
2,content_8e1334d6356668e3,client_73cda7b4e4f265ea,4-10,134984.0,0.000007,0.003239,0.003232,436.206806,CTR_GAP_VS_POSITION_TIER,review_ctr
3,content_34a70fea29d15f24,client_62f4a7e64f5e0096,4-10,143019.0,0.000301,0.003239,0.002938,420.231792,CTR_GAP_VS_POSITION_TIER,review_ctr
4,content_fec55986a1868d62,client_73cda7b4e4f265ea,4-10,124075.0,0.000008,0.003239,0.003231,400.873070,CTR_GAP_VS_POSITION_TIER,review_ctr
5,content_7c6373141eae744a,client_62f4a7e64f5e0096,4-10,132593.0,0.000626,0.003239,0.002613,346.462470,CTR_GAP_VS_POSITION_TIER,review_ctr
6,content_f6116743b00afc2d,client_62f4a7e64f5e0096,4-10,107584.0,0.000139,0.003239,0.003100,333.459499,CTR_GAP_VS_POSITION_TIER,review_ctr
7,content_306bc78dff1eb683,client_e547b89c05043229,1-3,80821.0,0.000433,0.004063,0.003629,293.337767,CTR_GAP_VS_POSITION_TIER,review_ctr
8,content_acbcc847f8996314,client_62f4a7e64f5e0096,4-10,170808.0,0.001534,0.003239,0.001705,291.239052,CTR_GAP_VS_POSITION_TIER,review_ctr
9,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,4-10,89332.0,0.000045,0.003239,0.003194,285.342132,CTR_GAP_VS_POSITION_TIER,review_ctr


## 3. Top-10 review

For each of the top ten: the action, why it's there, and what would make it wrong.

In [16]:
top10 = ranked.head(10).copy()

for i, row in top10.iterrows():
    print(f"#{i+1} — content_hash_id={row['content_hash_id'][:12]}... "
          f"(client={row['client_hash_id'][:12]}...)")
    print(f"  Action: {row['action_label']}  |  Reason: {row['reason_code']}")
    print(f"  Position tier: {row['position_tier']}  |  Impressions: {row['total_impressions']:.0f}  |  "
          f"CTR: {row['avg_ctr']:.4f} vs expected {row['expected_ctr']:.4f} (gap {row['ctr_gap']:.4f})")
    print(f"  Score: {row['score']:.1f}")
    print()

#1 — content_hash_id=content_44f3... (client=client_23a62...)
  Action: review_ctr  |  Reason: CTR_GAP_VS_POSITION_TIER
  Position tier: 4-10  |  Impressions: 212404  |  CTR: 0.0001 vs expected 0.0032 (gap 0.0031)
  Score: 664.0

#2 — content_hash_id=content_8d7d... (client=client_e547b...)
  Action: review_ctr  |  Reason: CTR_GAP_VS_POSITION_TIER
  Position tier: 1-3  |  Impressions: 203497  |  CTR: 0.0014 vs expected 0.0041 (gap 0.0026)
  Score: 537.7

#3 — content_hash_id=content_8e13... (client=client_73cda...)
  Action: review_ctr  |  Reason: CTR_GAP_VS_POSITION_TIER
  Position tier: 4-10  |  Impressions: 134984  |  CTR: 0.0000 vs expected 0.0032 (gap 0.0032)
  Score: 436.2

#4 — content_hash_id=content_34a7... (client=client_62f4a...)
  Action: review_ctr  |  Reason: CTR_GAP_VS_POSITION_TIER
  Position tier: 4-10  |  Impressions: 143019  |  CTR: 0.0003 vs expected 0.0032 (gap 0.0029)
  Score: 420.2

#5 — content_hash_id=content_fec5... (client=client_73cda...)
  Action: review_ct

1. **Action:** review_ctr. **Why:** content_44f3 has 212,404 impressions at tier 4-10, but CTR is 0.0001 vs expected 0.0032 (gap 0.0031) — the single largest visible-volume gap in the whole slice. **What would make it wrong:** if this page's near-zero CTR reflects an intent mismatch (e.g. an informational query landing on a transactional page) rather than a fixable snippet/title issue.

2. **Action:** review_ctr. **Why:** content_8d7d has 203,497 impressions at tier 1-3, CTR 0.0014 vs expected 0.0041 (gap 0.0026) — even at a top-3 position it's underperforming its tier's typical CTR. **What would make it wrong:** if the page already ranks #1-3 for a highly competitive query where a low CTR is expected because a SERP feature (featured snippet, ads) is taking the clicks instead of any organic result.

3. **Action:** review_ctr. **Why:** content_8e13 has 134,984 impressions at tier 4-10, CTR is exactly 0.0000 vs expected 0.0032 (gap 0.0032) — zero clicks despite huge visibility. **What would make it wrong:** if this page's zero clicks is a tracking/measurement gap (e.g. clicks not firing correctly) rather than a genuine snippet problem — worth a tracking sanity check before assuming it's a content fix.

4. **Action:** review_ctr. **Why:** content_34a7 has 143,019 impressions at tier 4-10, CTR 0.0003 vs expected 0.0032 (gap 0.0029). **What would make it wrong:** if the query intent behind these impressions is broad/navigational and users aren't expected to click through regardless of snippet quality.

5. **Action:** review_ctr. **Why:** content_fec5 has 124,075 impressions at tier 4-10, CTR 0.0000 vs expected 0.0032 (gap 0.0032) — same zero-click pattern as #3. **What would make it wrong:** same tracking-gap risk as #3 — worth checking whether this is a data collection issue rather than a real content problem before prioritizing a rewrite.

6. **Action:** review_ctr. **Why:** content_7c63 has 132,593 impressions at tier 4-10, CTR 0.0006 vs expected 0.0032 (gap 0.0026). **What would make it wrong:** if this page recently changed URL/redirected and GSC data is still catching up, making the CTR figure temporarily unreliable.

7. **Action:** review_ctr. **Why:** content_f611 has 107,584 impressions at tier 4-10, CTR 0.0001 vs expected 0.0032 (gap 0.0031). **What would make it wrong:** if the impressions are concentrated on a single unusual query that isn't representative of the page's main topic, inflating impressions without a realistic path to more clicks.

8. **Action:** review_ctr. **Why:** content_306b has 80,821 impressions at tier 1-3, CTR 0.0004 vs expected 0.0041 (gap 0.0036) — the largest relative gap of any top-3-tier page in the list. **What would make it wrong:** same SERP-feature risk as #2 — a top-3 ranking with heavy snippet competition (ads, People Also Ask) can suppress CTR regardless of title/meta quality.

9. **Action:** review_ctr. **Why:** content_acbc has 170,808 impressions at tier 4-10, CTR 0.0015 vs expected 0.0032 (gap 0.0017) — the smallest gap in the top-10, meaning it's here mostly because of its sheer impression volume rather than a severe underperformance. **What would make it wrong:** if the volume-weighted score is over-prioritizing this page simply for its scale — a smaller page with a proportionally bigger gap might deserve attention first.

10. **Action:** review_ctr. **Why:** content_cd3d has 89,332 impressions at tier 4-10, CTR 0.0000 vs expected 0.0032 (gap 0.0032). **What would make it wrong:** same tracking-gap concern as #3 and #5 — three of the top-10 picks share this exact zero-CTR-despite-high-impressions pattern, which is worth checking as a possible systematic issue (e.g. a template or content type where click tracking silently fails) rather than treating each as an independent content problem.

## 4. Weak picks + leakage check

Which picks look wrong, and confirmation that no product flags or future-window inputs leaked into the score.

In [17]:
# Weak-pick scan: rows near the top whose position tier's own bucket had thin support (n below the
# ~50-row floor), or whose expected_ctr is itself unstable because the tier sample was small.
tier_n = df.groupby("position_tier", observed=True).size()
print("n per position tier (used to compute expected_ctr):")
print(tier_n)

thin_tiers = tier_n[tier_n < 50].index.tolist()
weak_in_top10 = top10[top10["position_tier"].isin(thin_tiers)]
print(f"\nTop-10 rows resting on a thin tier (n < 50): {len(weak_in_top10)}")
print(weak_in_top10[["content_hash_id", "position_tier", "score"]] if len(weak_in_top10) else "None — all top-10 rows rest on tiers with sufficient support.")

# --- Leakage check ---
print("\n[Leakage check] Inputs to the score: total_impressions, avg_ctr, expected_ctr (a same-month",
      "tier median), position_tier — all computed from the SAME month, no future window used.")
print("[Leakage check] No product flags or existing-system scores used as inputs — none exist in",
      "this table; expected_ctr is derived entirely in this notebook from this month's own data.")
print("[Leakage check] No label was used at all in this baseline (it's a rule, not a trained model),",
      "so there is no label-derived-feature risk in the sense ML-04/05 tested for.")
print("[Leakage check] Sample-size floor (~50 rows) checked per position tier above before trusting",
      "any tier's expected_ctr.")

n per position tier (used to compute expected_ctr):
position_tier
1-3      16144
4-10     81987
11-20    32204
21-50    33288
51+      11681
dtype: int64

Top-10 rows resting on a thin tier (n < 50): 0
None — all top-10 rows rest on tiers with sufficient support.

[Leakage check] Inputs to the score: total_impressions, avg_ctr, expected_ctr (a same-month tier median), position_tier — all computed from the SAME month, no future window used.
[Leakage check] No product flags or existing-system scores used as inputs — none exist in this table; expected_ctr is derived entirely in this notebook from this month's own data.
[Leakage check] No label was used at all in this baseline (it's a rule, not a trained model), so there is no label-derived-feature risk in the sense ML-04/05 tested for.
[Leakage check] Sample-size floor (~50 rows) checked per position tier above before trusting any tier's expected_ctr.


Weak picks found: none by the thin-tier test — all top-10 rows rest on tiers with n ≥ 11,681. However, worth flagging as a judgment call: 3 of the top-10 picks (#3, #5, #10) share an identical pattern — CTR exactly 0.0000 at tier 4-10 with over 100k impressions each — which is more consistent with a possible tracking/measurement issue than three independent content problems, and should be checked before assuming a content fix will help. Separately, rows with avg_ctr = 0.0000 and huge impressions dominate the top of the list purely because of scale, not necessarily because they're the most "fixable" — a smaller page with a similarly-sized relative gap won't surface. This is a known bias of a volume-weighted score, not a leakage issue. All score inputs are same-month and non-label-derived, so this baseline is safe from the leakage patterns tested in ML-04/05 by construction — there simply isn't a label or a future window in this rule to leak from.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.